In [1]:
# =============================================================================
#  Angrist & Evans (1998) replication - multi-era harmonized pipeline
#  Extracts:  usa_00001 = ACS 2021-24 (1-yr)  |  usa_00002 = Census 1980 & 1990 5% state
#  The SAME code path runs on every era, so differences reflect TIME, not method.
# =============================================================================
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

from ipumspy import readers
import statsmodels.api as sm
from linearmodels.iv import IV2SLS

# extract stub -> human label
EXTRACTS = {
    "usa_00001": "ACS 2021-24",
    "usa_00002": "Census 1980/90",
}

# baseline controls (year fixed effects are added automatically when an era pools years)
CONTROLS_BASE = ['AGE', 'age_at_first_birth', 'boy_1st', 'boy_2nd',
                 'black', 'hispanic', 'other_race']

print("Environment ready - statsmodels + linearmodels + ipumspy loaded.")

Environment ready - statsmodels + linearmodels + ipumspy loaded.


In [2]:
# =============================================================================
#  DATA ENGINE - one function, applied identically to every extract / era
# =============================================================================

def load_households(stub, chunksize=100000):
    """Stream an IPUMS extract and keep every member of any household that
    contains a woman aged 21-35.  Matching is on (YEAR, SERIAL) so the
    repeated-SERIAL-across-years problem cannot contaminate households."""
    ddi = readers.read_ipums_ddi(f"{stub}.xml")
    kept = []
    for chunk in readers.read_microdata_chunked(ddi, f"{stub}.dat", chunksize=chunksize):
        women = chunk[(chunk['SEX'] == 2) & (chunk['AGE'].between(21, 35))]
        keys = women[['YEAR', 'SERIAL']].drop_duplicates()
        kept.append(chunk.merge(keys, on=['YEAR', 'SERIAL'], how='inner'))
    return pd.concat(kept, axis=0, ignore_index=True)


def build_panel(stub):
    """Return a compact mother-level panel: treatment, instruments,
    A&E-harmonized outcomes, husband's labor supply, controls and PERWT weight."""
    label = EXTRACTS[stub]
    print(f"  building {label} ({stub}) ...")
    raw = load_households(stub)

    # globally unique household-member id (YEAR breaks the SERIAL collision)
    def uid(d, loc):
        return (d['YEAR'].astype(str) + "_" + d['SERIAL'].astype(str) + "_"
                + d[loc].fillna(0).astype(int).astype(str))

    # --- mothers ---------------------------------------------------------
    moms = raw[(raw['SEX'] == 2) & (raw['AGE'].between(21, 35))].copy()
    moms['mom_id'] = uid(moms, 'PERNUM')

    # --- children, ordered by age to recover the first two births --------
    kids = raw[raw['MOMLOC'] > 0].copy()
    kids['mom_id'] = uid(kids, 'MOMLOC')
    kids = kids.sort_values(['mom_id', 'AGE', 'PERNUM'], ascending=[True, False, True])
    kids['birth_order'] = kids.groupby('mom_id').cumcount() + 1
    kid1 = kids[kids.birth_order == 1].set_index('mom_id')[['AGE', 'SEX']].rename(columns={'AGE': 'age_k1', 'SEX': 'sex_k1'})
    kid2 = kids[kids.birth_order == 2].set_index('mom_id')[['AGE', 'SEX']].rename(columns={'AGE': 'age_k2', 'SEX': 'sex_k2'})
    total_kids = kids.groupby('mom_id').size().rename('total_kids')

    df = moms.join(kid1, on='mom_id').join(kid2, on='mom_id').join(total_kids, on='mom_id')

    # --- A&E sample restrictions ----------------------------------------
    df = df[(df['total_kids'] >= 2) & (df['age_k1'] < 18)].copy()
    df['age_at_first_birth'] = df['AGE'] - df['age_k1']
    df = df[df['age_at_first_birth'] >= 15].copy()

    # --- treatment & instruments ----------------------------------------
    df['more_than_2'] = (df['total_kids'] > 2).astype(int)
    df['boy_1st']   = (df['sex_k1'] == 1).astype(int)
    df['boy_2nd']   = (df['sex_k2'] == 1).astype(int)
    df['same_sex']  = (df['boy_1st'] == df['boy_2nd']).astype(int)
    df['two_boys']  = ((df['boy_1st'] == 1) & (df['boy_2nd'] == 1)).astype(int)
    df['two_girls'] = ((df['boy_1st'] == 0) & (df['boy_2nd'] == 0)).astype(int)

    # --- husband's weeks worked (married sample) ------------------------
    df['spouse_id'] = uid(df, 'SPLOC')
    men = raw[(raw['SEX'] == 1) & (raw['AGE'] >= 18)].copy()
    men['person_id'] = uid(men, 'PERNUM')
    husbands = men[['person_id', 'WKSWORK1']].rename(columns={'WKSWORK1': 'husband_weeks'})
    df = df.merge(husbands, left_on='spouse_id', right_on='person_id', how='left')

    # --- outcomes (A&E definitions, harmonized across eras) --------------
    df['weeks_worked']   = pd.to_numeric(df['WKSWORK1'], errors='coerce').fillna(0.0)
    df['worked_for_pay'] = (df['weeks_worked'] > 0).astype(float)            # CHANGE: worked LAST YEAR (not EMPSTAT)
    df['hours_week']     = pd.to_numeric(df['UHRSWORK'], errors='coerce').fillna(0.0)
    inc = pd.to_numeric(df['INCWAGE'], errors='coerce')
    inc = inc.where(inc < 999998, 0.0).fillna(0.0)                          # 999998/9 = N/A
    df['labor_income']   = inc * pd.to_numeric(df['CPI99'], errors='coerce')  # CONSTANT 1999 $ -> cross-era comparable
    df['husband_weeks']  = pd.to_numeric(df['husband_weeks'], errors='coerce')

    # --- controls & weight ----------------------------------------------
    df['black']      = (df['RACE'] == 2).astype(float)
    df['hispanic']   = (df['HISPAN'] > 0).astype(float)
    df['other_race'] = ((df['RACE'] > 2) & (df['HISPAN'] == 0)).astype(float)
    df['weight']     = pd.to_numeric(df['PERWT'], errors='coerce').fillna(0.0)  # CHANGE: PERWT weighting
    df['married']    = (df['SPLOC'] > 0).astype(int)

    cols = ['YEAR', 'mom_id', 'weight', 'married',
            'more_than_2', 'same_sex', 'two_boys', 'two_girls', 'boy_1st', 'boy_2nd',
            'AGE', 'age_at_first_birth', 'black', 'hispanic', 'other_race',
            'worked_for_pay', 'weeks_worked', 'hours_week', 'labor_income', 'husband_weeks']
    out = df[cols].copy()

    # --- internal sanity asserts (catch construction bugs early) --------
    assert out['mom_id'].is_unique, "mother ids not unique -> husband merge duplicated rows"
    sr, ss = out['boy_1st'].mean(), out['same_sex'].mean()
    assert 0.50 <= sr <= 0.53, f"firstborn sex ratio off ({sr:.3f}) -> child linking broken"
    assert 0.49 <= ss <= 0.52, f"same-sex share off ({ss:.3f})"
    print(f"    -> {len(out):,} mothers | firstborn-boy {sr:.3f} | same-sex {ss:.3f}  [checks OK]")
    return out

In [3]:
# =============================================================================
#  BUILD ALL ERAS (same pipeline, one call per extract) then POOL
#  Result is cached to panel_all.pkl -> delete that file to force a rebuild
#  after changing build_panel().
# =============================================================================
import os
CACHE = "panel_all.pkl"
if os.path.exists(CACHE):
    df_all = pd.read_pickle(CACHE)
    print(f"loaded cached panel: {len(df_all):,} mothers")
else:
    panels = [build_panel(stub) for stub in EXTRACTS]   # usa_00001 (ACS), usa_00002 (1980+1990)
    df_all = pd.concat(panels, axis=0, ignore_index=True)
    df_all = df_all[df_all['weight'] > 0].reset_index(drop=True)  # IV2SLS needs strictly positive weights
    df_all.to_pickle(CACHE)

# era label: the four ACS years are pooled into one period
df_all['era'] = np.select(
    [df_all['YEAR'] <= 1980, df_all['YEAR'] <= 1990],
    ['1980', '1990'],
    default='2021-24')
ERAS = ['1980', '1990', '2021-24']

print("\nMothers per era (after all A&E restrictions):")
print(df_all.groupby('era').size().reindex(ERAS).to_string())

  building ACS 2021-24 (usa_00001) ...
    -> 237,548 mothers | firstborn-boy 0.514 | same-sex 0.506  [checks OK]
  building Census 1980/90 (usa_00002) ...
    -> 1,058,659 mothers | firstborn-boy 0.512 | same-sex 0.505  [checks OK]

Mothers per era (after all A&E restrictions):
era
1980       528324
1990       530295
2021-24    237548


In [4]:
# =============================================================================
#  TABLE 3 - fraction that had another child, by parity & sex of first two
#  PERWT-weighted.  All paper categories retained, per era and by marital status.
# =============================================================================

def _wprop(d, mask):
    """PERWT-weighted P(more_than_2) on a subgroup, with Kish-effective-N s.e.
    and the subgroup's weighted share of the sample."""
    s = d[mask]
    w = s['weight'].to_numpy(dtype=float)
    if w.sum() == 0:
        return np.nan, np.nan, np.nan
    p = np.average(s['more_than_2'], weights=w)
    n_eff = w.sum() ** 2 / np.square(w).sum()
    se = np.sqrt(p * (1 - p) / n_eff)
    return p, se, w.sum() / d['weight'].sum()

def table3(d):
    cats = [("one boy, one girl", d['same_sex'] == 0),
            ("two girls",         d['two_girls'] == 1),
            ("two boys",          d['two_boys'] == 1),
            ("both same sex",     d['same_sex'] == 1)]
    rows, store = {}, {}
    for name, mask in cats:
        p, se, fr = _wprop(d, mask)
        store[name] = (p, se)
        rows[name] = [f"{p:.3f}", f"({se:.3f})", f"{fr:.3f}"]
    pm, sem = store["one boy, one girl"]
    ps, ses = store["both same sex"]
    diff, sed = ps - pm, np.sqrt(sem ** 2 + ses ** 2)
    rows["difference (same - mixed)"] = [f"{diff:.3f}", f"({sed:.3f})", ""]
    return pd.DataFrame(rows, index=["had another child", "  (s.e.)", "frac of sample"]).T

print("=" * 70)
print("TABLE 3  -  FRACTION THAT HAD ANOTHER CHILD  (PERWT-weighted)")
print("=" * 70)
for era in ERAS:
    d = df_all[df_all['era'] == era]
    for sample, sub in [("All women", d), ("Married women", d[d['married'] == 1])]:
        print(f"\n[{era}]  {sample}   (n = {len(sub):,}, weighted N = {int(sub['weight'].sum()):,})")
        print(table3(sub).to_string())
print("\n" + "=" * 70)

TABLE 3  -  FRACTION THAT HAD ANOTHER CHILD  (PERWT-weighted)

[1980]  All women   (n = 528,324, weighted N = 10,566,480)
                          had another child   (s.e.) frac of sample
one boy, one girl                     0.344  (0.001)          0.495
two girls                             0.403  (0.001)          0.241
two boys                              0.388  (0.001)          0.263
both same sex                         0.395  (0.001)          0.505
difference (same - mixed)             0.051  (0.001)               

[1980]  Married women   (n = 440,794, weighted N = 8,815,880)
                          had another child   (s.e.) frac of sample
one boy, one girl                     0.338  (0.001)          0.495
two girls                             0.401  (0.002)          0.239
two boys                              0.385  (0.001)          0.266
both same sex                         0.392  (0.001)          0.505
difference (same - mixed)             0.055  (0.001)               

In [5]:
# =============================================================================
#  TABLE 7 - OLS & 2SLS effect of (>2 children) on labor supply, PERWT-weighted
#  Estimators per outcome:  OLS | 2SLS same-sex | 2SLS two-boys/two-girls
# =============================================================================
RESULTS = {}   # (era, sample, outcome) -> estimates, consumed by the cross-era cell

def _design(d_in):
    """Add year fixed effects when an era pools multiple survey years (ACS)."""
    d = d_in.copy()
    ctrl = list(CONTROLS_BASE)
    if d['YEAR'].nunique() > 1:
        yd = pd.get_dummies(d['YEAR'], prefix='yr', drop_first=True).astype(float)
        d = pd.concat([d, yd], axis=1)
        ctrl += list(yd.columns)
    return d, ctrl

def run_table7(d_in, outcomes, era, sample):
    d, ctrl = _design(d_in)
    d = d.dropna(subset=outcomes + ['more_than_2', 'same_sex', 'two_boys', 'two_girls', 'weight'] + ctrl)
    w = d['weight']
    X = sm.add_constant(d[ctrl].astype(float))
    endog = d['more_than_2'].astype(float)
    fmt = {}
    for y in outcomes:
        Y = d[y].astype(float)
        # OLS (weighted, robust)
        m_ols = sm.WLS(Y, X.assign(more_than_2=endog), weights=w).fit(cov_type='HC1')
        b_ols, s_ols = m_ols.params['more_than_2'], m_ols.bse['more_than_2']
        # 2SLS - same-sex instrument
        m1 = IV2SLS(Y, X, endog, d['same_sex'].astype(float), weights=w).fit(cov_type='robust')
        b1, s1 = m1.params['more_than_2'], m1.std_errors['more_than_2']
        # 2SLS - two-boys / two-girls instruments (boy_2nd drops: collinear with instruments)
        Xc = X.drop(columns=['boy_2nd'])
        m2 = IV2SLS(Y, Xc, endog, d[['two_boys', 'two_girls']].astype(float), weights=w).fit(cov_type='robust')
        b2, s2 = m2.params['more_than_2'], m2.std_errors['more_than_2']
        try:
            fsF = float(m1.first_stage.diagnostics['f.stat'].iloc[0])
        except Exception:
            fsF = np.nan
        RESULTS[(era, sample, y)] = dict(ols=(b_ols, s_ols), iv_ss=(b1, s1), iv_bg=(b2, s2), fsF=fsF)
        fmt[y] = [f"{b_ols:.3f} ({s_ols:.3f})", f"{b1:.3f} ({s1:.3f})", f"{b2:.3f} ({s2:.3f})"]
    return pd.DataFrame(fmt, index=["OLS", "2SLS (same sex)", "2SLS (2boy/2girl)"]).T

OUT_ALL = ['worked_for_pay', 'weeks_worked', 'hours_week', 'labor_income']
OUT_MAR = OUT_ALL + ['husband_weeks']

print("=" * 78)
print("TABLE 7  -  EFFECT OF >2 CHILDREN ON LABOR SUPPLY  (PERWT-weighted)")
print("           labor_income in constant 1999 dollars (CPI99-deflated)")
print("=" * 78)
for era in ERAS:
    d = df_all[df_all['era'] == era]
    for sample, sub, outs in [("All women", d, OUT_ALL),
                              ("Married women", d[d['married'] == 1], OUT_MAR)]:
        print(f"\n[{era}]  {sample}   (n = {len(sub):,})")
        print(run_table7(sub, outs, era, sample).to_string())
print("\n" + "=" * 78)

TABLE 7  -  EFFECT OF >2 CHILDREN ON LABOR SUPPLY  (PERWT-weighted)
           labor_income in constant 1999 dollars (CPI99-deflated)

[1980]  All women   (n = 528,324)
                               OLS      2SLS (same sex)    2SLS (2boy/2girl)
worked_for_pay      -0.171 (0.001)       -0.105 (0.026)       -0.098 (0.026)
weeks_worked        -8.578 (0.064)       -4.834 (1.131)       -4.555 (1.124)
hours_week          -6.546 (0.056)       -3.564 (0.981)       -3.303 (0.975)
labor_income    -3911.148 (33.120)  -1734.416 (590.329)  -1665.491 (587.464)

[1980]  Married women   (n = 440,794)
                               OLS      2SLS (same sex)    2SLS (2boy/2girl)
worked_for_pay      -0.164 (0.002)       -0.117 (0.026)       -0.112 (0.026)
weeks_worked        -8.011 (0.069)       -5.299 (1.139)       -5.108 (1.132)
hours_week          -6.075 (0.060)       -4.187 (0.983)       -3.997 (0.976)
labor_income    -3533.265 (35.249)  -1755.999 (584.144)  -1749.596 (580.677)
husband_weeks       -0

In [9]:
# =============================================================================
#  WHAT CHANGED IN 30 YEARS - formal comparison of the headline estimates
#  Independent era-samples =>  D = b_end - b_base ,  s.e.(D) = sqrt(se_base^2 + se_end^2)
#  -> simple z-test on every coefficient.
# =============================================================================

def compare(sample, outcomes, estimator='iv_ss', base='1980', end='2021-24'):
    rows = {}
    for y in outcomes:
        a, b = RESULTS.get((base, sample, y)), RESULTS.get((end, sample, y))
        if a is None or b is None:
            continue
        (ba, sa), (bb, sb) = a[estimator], b[estimator]
        D, se = bb - ba, np.sqrt(sa ** 2 + sb ** 2)
        z = D / se if se else np.nan
        rows[y] = [f"{ba:.3f}", f"{bb:.3f}", f"{D:+.3f}", f"({se:.3f})", f"{z:+.2f}"]
    return pd.DataFrame(rows, index=[base, end, f"D ({end}-{base})", "  s.e.", "z"]).T

print("=" * 74)
print("WHAT CHANGED 1980 -> 2021-24   (2SLS same-sex LATE, PERWT-weighted)")
print("=" * 74)
print("\nMarried women:")
print(compare("Married women", OUT_MAR).to_string())
print("\nAll women:")
print(compare("All women", OUT_ALL).to_string())

# channel: did the instrument weaken? (first-stage F of same-sex across eras)
print("\n" + "=" * 74)
print("INSTRUMENT STRENGTH over time  (first-stage F, same-sex, married sample)")
print("=" * 74)
fs = {era: [f"{RESULTS[(era, 'Married women', 'worked_for_pay')]['fsF']:.0f}"]
      for era in ERAS if (era, 'Married women', 'worked_for_pay') in RESULTS}
print(pd.DataFrame(fs, index=["first-stage F"]).T.to_string())
print("=" * 74)

WHAT CHANGED 1980 -> 2021-24   (2SLS same-sex LATE, PERWT-weighted)

Married women:
                     1980    2021-24 D (2021-24-1980)        s.e.      z
worked_for_pay     -0.117     -0.117           -0.001     (0.064)  -0.01
weeks_worked       -5.299     -7.629           -2.330     (3.192)  -0.73
hours_week         -4.187     -5.298           -1.111     (2.577)  -0.43
labor_income    -1755.999  -2958.277        -1202.279  (2752.548)  -0.44
husband_weeks       0.183      2.124           +1.941     (1.895)  +1.02

All women:
                     1980    2021-24 D (2021-24-1980)        s.e.      z
worked_for_pay     -0.105     -0.166           -0.062     (0.064)  -0.97
weeks_worked       -4.834     -8.958           -4.124     (3.248)  -1.27
hours_week         -3.564     -7.005           -3.442     (2.622)  -1.31
labor_income    -1734.416  -4793.770        -3059.354  (2639.912)  -1.16

INSTRUMENT STRENGTH over time  (first-stage F, same-sex, married sample)
        first-stage F
1980 